## 16_config_pattern — real-world config pattern demo (C360-style).

Run in EITHER workspace AFTER 00_setup_sample_data (queries practice_customers;
the bootstrap section stages + installs the practice requirements).

Covers the three real-world errors this project's lib fixes:
 1) CFG = setup_config(config_path=CONFIG_PATH)
    -> TypeError: setup_config() missing 1 required positional argument: 'run_date'
    Fix: pass run_date (widget/job param): setup_config(config_path=..., run_date=...)
 2) spark.sql(f"SHOW PARTITIONS {tbl}").orderBy("event_partition_date", ascending=False)
    -> PySparkValueError (no such column: SHOW PARTITIONS returns ONE string
       column named `partition`, e.g. 'event_partition_date=2026-10-18')
    Fix: regexp-extract the date first, then order desc (see latest_partition()).
 3) %pip install -r "$REQ_PATH" -q
    -> ERROR: Invalid requirement: '.../requirements.txt' (the magic word-splits
       a project path containing spaces; pip receives the tail as a requirement)
    Fix: stage requirements at a space-free path first (prepare_requirements()).

Note: the catalog-driven CFG (CATALOG/MODEL_CATALOG widgets, dev/stg/prd
stage catalogs, 3-part table names) is the NEW-platform pattern — it lives
in bundle/src/lib/config_lib.py. The OLD workspace has NO catalog for
tables: 2-part schema.table names only (models/volumes excepted).


In [ ]:

%run ./lib/config_lib


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")
dbutils.widgets.text("PARTITION", "latest", "all | latest | prev | YYYY-MM-DD")
dbutils.widgets.text("CONFIG_PATH", "", "Optional env config file (empty = built-in dev/stg/prd)")
SCHEMA_WIDGET = dbutils.widgets.get("SCHEMA").strip()
PARTITION_WIDGET = dbutils.widgets.get("PARTITION").strip()
CONFIG_PATH = dbutils.widgets.get("CONFIG_PATH").strip()


In [ ]:

# The CORRECT call — run_date (= PARTITION widget) is always passed:
CFG = setup_config(
    config_path=CONFIG_PATH,
    run_date=PARTITION_WIDGET,
    schema=SCHEMA_WIDGET,
)

# This is the call that produced the real-world TypeError (kept as a comment):
# CFG = setup_config(config_path=CONFIG_PATH)   # <- missing run_date

print("catalog    : none — old workspace uses 2-part schema.table names")
print(f"schema     : {CFG.schema}")
print(f"run_date   : {CFG.run_date}")


In [ ]:

print("=== CFG-RESOLVED TABLE NAMES (schema.table — old workspace, no catalog) ===")
for t in ["practice_customers", "practice_customers_clean", "practice_customers_enriched",
          "practice_customers_summary"]:
    print(f"  {CFG.table(t)}")
print()
print("=== table naming on this workspace (OLD) ===")
print(f"  tables -> {CFG.table('practice_customers')}   (2-part: schema.table, no catalog)")


In [ ]:

# Partition discovery — the corrected SHOW PARTITIONS pattern.
tbl = CFG.table("practice_customers")

latest = latest_partition(tbl, k=0)
prev = latest_partition(tbl, k=1)
print(f"latest partition (k=0): {latest}")
print(f"prev partition    (k=1): {prev}")
print("(k=1 mirrors the real code's `.collect()[1]` — skipping the in-flight newest date)")


In [ ]:

# run_date='prev' in action: CFG.partition_filter() resolves latest/prev at runtime.
CFG_prev = setup_config(config_path=CONFIG_PATH, run_date="prev", schema=SCHEMA_WIDGET)

n_prev = CFG_prev.partition_filter(spark.read.table(tbl), table=tbl).count()
n_all = spark.read.table(tbl).count()
print(f"rows total           : {n_all}")
print(f"rows prev partition  : {n_prev}  (run_date='prev' -> ingest_date={prev})")


In [ ]:

print("=== PROJECT BOOTSTRAP (old-school requirements install) ===")
# Production shape, kept verbatim:
#   %pip install -r "$REQ_PATH" -q
# Why it failed in production: the project path had spaces ('Shared/DS Use
# Case/...'), the %pip magic word-splits the interpolated value on whitespace,
# and pip received 'Case/biap-.../requirements.txt' as a package REQUIREMENT:
#   ERROR: Invalid requirement: 'Case/.../requirements.txt'
# Fix (line unchanged): stage requirements at a SPACE-FREE absolute path first.

import os

PROJECT_PATH = setup_wd_path(dbutils)
REQ_PATH = prepare_requirements(PROJECT_PATH)
print(f"PROJECT_PATH : {PROJECT_PATH}")
print(f"REQ_PATH     : {REQ_PATH}  (space-free -> the old-school line works)")


In [ ]:

%pip install -r "$REQ_PATH" -q


In [ ]:

import xgboost
print(f"xgboost installed: {xgboost.__version__}")


In [ ]:

print("=== SUMMARY ===")
print("- setup_config(config_path, run_date, ...) — run_date is required; pass it from a widget/job param.")
print("- latest_partition(table, k) — SHOW PARTITIONS returns one `partition` STRING column;")
print("  regexp-extract the date before ordering; k=0 latest, k=1 prev.")
print("- prepare_requirements() stages requirements at a space-free /tmp path, so the")
print("  production line %pip install -r \"$REQ_PATH\" -q works unchanged (Appendix A.4).")
print("- The old workspace has NO catalog for tables (2-part schema.table names).")
print("  The full catalog CFG (dev/stg/prd, MODEL_CATALOG, 3-part names) lives on the")
print("  NEW platform: bundle/src/lib/config_lib.py.")
